In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

all_sheets = pd.read_excel('/content/drive/MyDrive/Cadetx /ev_charging_dataset.xlsx', sheet_name=None)
sessions_df = all_sheets['sessions']
stations_df = all_sheets['stations']

print(sessions_df[['station_id', 'station_region', 'energy_kwh']].head())

  station_id station_region  energy_kwh
0    STN_001     North West        56.9
1    STN_001     North West        40.7
2    STN_001     North West        30.9
3    STN_001     North West        28.4
4    STN_001     North West        31.7


**Total energy delivered per station**

In [3]:
station_energy = sessions_df.groupby('station_id')['energy_kwh'].sum().reset_index()
station_energy.columns = ['station_id', 'total_kwh_delivered']

print(station_energy.head())
print(station_energy.shape)

  station_id  total_kwh_delivered
0    STN_001             620951.6
1    STN_002             940999.0
2    STN_003             733802.6
3    STN_004             720723.5
4    STN_005             702048.5
(33, 2)


In [4]:
region_energy = sessions_df.groupby('station_region')['energy_kwh'].sum().reset_index()
region_energy.columns = ['region', 'total_kwh_delivered']

print(region_energy.sort_values('total_kwh_delivered', ascending=False))

               region  total_kwh_delivered
3          North West            3005135.1
1            Midlands            2576709.8
0              London            1978090.6
6          South West            1580496.2
2          North East            1398050.3
5          South East            1088495.1
4            Scotland             947010.5
8  Yorkshire & Humber             899734.0
7               Wales             335024.2


In [5]:
sessions_df['hour'] = sessions_df['start_timestamp'].dt.hour

hourly_energy = sessions_df.groupby('hour')['energy_kwh'].sum().reset_index()
hourly_energy.columns = ['hour', 'total_kwh']

print(hourly_energy)

    hour  total_kwh
0      0   128740.8
1      1   125309.7
2      2   127569.8
3      3   124141.6
4      4   127238.3
5      5   250529.5
6      6   375601.5
7      7   498779.7
8      8   750802.3
9      9  1004475.5
10    10  1134524.0
11    11  1129683.7
12    12  1123218.3
13    13  1131742.8
14    14  1131890.0
15    15  1004073.7
16    16   881564.0
17    17   748783.4
18    18   634131.4
19    19   496195.6
20    20   375641.0
21    21   252607.5
22    22   125868.9
23    23   125632.8


In [6]:
stations_per_region = stations_df.groupby('region').size().reset_index(name='station_count')
print(stations_per_region.sort_values('station_count', ascending=False))

               region  station_count
3          North West              7
0              London              5
1            Midlands              5
5          South East              4
2          North East              3
4            Scotland              3
6          South West              3
8  Yorkshire & Humber              2
7               Wales              1


In [7]:
region_summary = region_energy.merge(stations_per_region, on='region')
region_summary['avg_kwh_per_station'] = region_summary['total_kwh_delivered'] / region_summary['station_count']
print(region_summary.sort_values('avg_kwh_per_station', ascending=False))

               region  total_kwh_delivered  station_count  avg_kwh_per_station
6          South West            1580496.2              3        526832.066667
1            Midlands            2576709.8              5        515341.960000
2          North East            1398050.3              3        466016.766667
8  Yorkshire & Humber             899734.0              2        449867.000000
3          North West            3005135.1              7        429305.014286
0              London            1978090.6              5        395618.120000
7               Wales             335024.2              1        335024.200000
4            Scotland             947010.5              3        315670.166667
5          South East            1088495.1              4        272123.775000


In [9]:
station_energy.to_csv('/content/drive/MyDrive/Cadetx /station_energy.csv', index=False)
hourly_energy.to_csv('/content/drive/MyDrive/Cadetx /hourly_energy.csv', index=False)
region_summary.to_csv('/content/drive/MyDrive/Cadetx /region_energy_per_station.csv', index=False)

print("All saved!")

All saved!


In [10]:
sessions_df['date'] = sessions_df['start_timestamp'].dt.date
daily_energy = sessions_df.groupby('date')['energy_kwh'].sum().reset_index()
daily_energy.columns = ['ds', 'y']
daily_energy['ds'] = pd.to_datetime(daily_energy['ds'])

print(daily_energy.head())

          ds       y
0 2022-01-01  5024.4
1 2022-01-02  3861.3
2 2022-01-03  6250.3
3 2022-01-04  5620.4
4 2022-01-05  6104.1


In [11]:
from prophet import Prophet
model = Prophet()
model.fit(daily_energy)

INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


In [12]:
future = model.make_future_dataframe(periods=90)
forecast = model.predict(future)

print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

             ds          yhat    yhat_lower    yhat_upper
1181 2025-03-27  29534.187086  27911.676964  31217.349262
1182 2025-03-28  29616.824113  27931.912410  31298.632135
1183 2025-03-29  25942.888305  24412.342872  27570.597044
1184 2025-03-30  25950.910191  24265.112504  27584.791389
1185 2025-03-31  29574.035650  27839.707784  31163.539314
